# Streaming and Async

Make LLM calls non-blocking, stream tokens as they arrive, and process batches concurrently.

In this tutorial you take the feedback analysis pipeline from Tutorial 01 and make it production-ready: non-blocking async calls, token-by-token streaming to a UI, and concurrent batch processing.

By the end you will have covered:

* `ainstruct()` and the async session method naming convention
* Streaming output with `ModelOption.STREAM` and `astream()`
* Concurrent fan-out with `wait_for_all_mots()`
* Mixing parallel and sequential dependencies in one pipeline
* How `ChatContext` affects async execution

Prerequisites: Quick Start complete, Mellea installed (`uv add mellea`), and a working backend such as Ollama.

---

## Step 1: Your first async call

The async API mirrors the synchronous API closely. Every major `MelleaSession` method has an `a`-prefixed async counterpart with the same core purpose and a similar signature. That means if you already understand [`instruct()`](mellea_tutorials/your_first_generative_program.ipynb), the transition to [`ainstruct()`](mellea_tutorials/streaming_and_async.ipynb) is mostly about control flow rather than a new mental model.

**Key concepts:**

- [`await m.ainstruct(...)`](mellea_tutorials/streaming_and_async.ipynb) starts an LLM request without blocking the whole Python program.
- The awaited call returns a `ModelOutputThunk`, just like the sync version. The difference is that generation begins inside an async workflow.
- Converting the thunk with [`str(result)`](mellea_tutorials/streaming_and_async.ipynb) resolves the final text when it is ready.
- The same naming pattern applies across the API, for example `achat()`, `aact()`, `aquery()`, `atransform()`, and `avalidate()`.

This first example is intentionally minimal: one async function, one session, one instruction. In a notebook, run the coroutine with top-level [`await`](mellea_tutorials/streaming_and_async.ipynb) because Jupyter already has an active event loop. In a normal Python script, use [`asyncio.run()`](mellea_tutorials/streaming_and_async.ipynb) from an `if __name__ == "__main__":` block.

In [ ]:
from mellea.core.base import ModelOutputThunk


import asyncio

import mellea


async def main() -> None:
    """Run a single asynchronous instruction and print the result."""
    # Create a standard session. The async behaviour comes from the method we call,
    # not from a separate session type.
    m: mellea.MelleaSession = mellea.start_session()

    # ainstruct() is the async counterpart to instruct().
    # Awaiting it returns a ModelOutputThunk representing the in-flight generation.
    result: ModelOutputThunk[str] = await m.ainstruct(
        "Summarise this customer feedback in one sentence: "
        "The onboarding was confusing and took far too long. "
        "Support was helpful once I got through."
    )

    # Converting the thunk to str resolves the generated text.
    print(str(result))


# Notebook usage:
await main()

# Script usage:
# if __name__ == "__main__":
#     asyncio.run(main())

# Output will vary by model and configuration.


## Step 2: Streaming tokens

Async generation becomes much more useful when you can surface partial output immediately. Streaming is how you build responsive terminals, chat UIs, progress displays, or any workflow where waiting for the entire response would feel slow.

**Key concepts:**

- Enable streaming by passing [`ModelOption.STREAM: True`](mellea_tutorials/streaming_and_async.ipynb) in `model_options`.
- Keep a reference to the returned `ModelOutputThunk`; this object is your handle for polling stream progress.
- Use [`await mot.astream()`](mellea_tutorials/streaming_and_async.ipynb) to receive the next available chunk of text.
- Use [`mot.is_computed()`](mellea_tutorials/streaming_and_async.ipynb) to know when the full generation has finished.
- Append chunks as they arrive so you can both display them live and reconstruct the final string afterwards.

One practical rule matters here: stream from a thunk in exactly one coroutine. The tutorial warns against calling [`astream()`](mellea_tutorials/streaming_and_async.ipynb) concurrently on the same thunk, because chunk consumption is stateful. In other words, one producer, one consumer.

Notebook environments often do not repaint standard output on every tiny `print(..., flush=True)` token update, so token streaming can appear buffered or delayed even though the underlying chunks are arriving correctly. Some notebook runtimes also render `Markdown` display updates and widget output as empty or unreliable cells while the stream is in flight. Bobide appears to behave that way. In addition, some Mellea backends return streamed chunks that are empty during progress updates and only expose the final text through the thunk value at the end. For that reason, the notebook example below resolves the final value from the thunk after streaming completes, while the script example still demonstrates true live token streaming with `print(..., end="", flush=True)` when the backend/frontend combination supports it.

In [ ]:
from mellea.core.base import ModelOutputThunk


import asyncio

import mellea
from mellea.backends.model_options import ModelOption


async def stream_summary(feedback: str, live_output: bool = False) -> str:
    """Stream a summary token by token and return the fully assembled text.

    Args:
        feedback: Source text to summarise.
        live_output: When True, print each non-empty chunk immediately for
            terminal/script usage. When False, resolve and print the final thunk
            value at the end, which is more reliable in Bobide notebooks.
    """
    m: mellea.MelleaSession = mellea.start_session()

    # Start a streamed generation. The returned thunk can now be consumed chunk-by-chunk.
    mot: ModelOutputThunk[str] = await m.ainstruct(
        "Summarise this customer feedback in one sentence: {{text}}",
        user_variables={"text": feedback},
        model_options={ModelOption.STREAM: True},
    )

    streamed_chunks: list[str] = []

    # Keep consuming stream events until the thunk reports completion.
    while not mot.is_computed():
        chunk: str = await mot.astream()

        # Some backends emit empty progress chunks. Keep only meaningful text.
        if chunk:
            streamed_chunks.append(chunk)
            if live_output:
                print(chunk, end="", flush=True)

    # Resolve the final value from the thunk itself. This is the most reliable
    # source of the completed text in notebook environments such as Bobide.
    final_text: str = str(mot)

    if live_output:
        print()
    else:
        print(final_text)

    return final_text


# Notebook usage in Bobide:
await stream_summary(
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through.",
    live_output=False,
)

# Script usage with true live token streaming:
# if __name__ == "__main__":
#     asyncio.run(
#         stream_summary(
#             "The onboarding was confusing and took far too long. "
#             "Support was helpful once I got through.",
#             live_output=True,
#         )
#     )

# Tip: do not call mot.astream() from multiple coroutines on the same thunk.


## Step 3: Concurrent batch processing

Async is not only about UI responsiveness. It also lets you overlap multiple independent LLM calls so total runtime is closer to the slowest single request than the sum of all requests. This is the core throughput gain for batch work.

**Key concepts:**

- Fire multiple [`ainstruct()`](mellea_tutorials/streaming_and_async.ipynb) calls first, without immediately resolving each result.
- Each awaited `ainstruct()` returns a thunk that represents a generation already in flight.
- Store those thunks in a list, then resolve them together with [`wait_for_all_mots()`](mellea_tutorials/streaming_and_async.ipynb).
- This pattern is ideal when each prompt is independent and there are no ordering constraints between items.

The subtle but important detail is that this example does **not** call [`str(thunk)`](mellea_tutorials/streaming_and_async.ipynb) inside the loop. Doing so would serialize the workflow and throw away the concurrency benefit. First launch everything, then wait once. In the notebook, assign the awaited result directly; in a script, wrap the same coroutine with [`asyncio.run()`](mellea_tutorials/streaming_and_async.ipynb).

Unlike Step 2, these examples do not need live token-by-token UI updates. Notebook and script behaviour should therefore be effectively the same here: wait for the async work to complete, then print final values.

In [ ]:
from mellea.core.base import ModelOutputThunk


import asyncio

import mellea
from mellea.helpers.async_helpers import wait_for_all_mots


async def summarise_batch(items: list[str]) -> list[str]:
    """Summarise many feedback items concurrently."""
    m: mellea.MelleaSession = mellea.start_session()

    # Launch all requests as quickly as possible.
    # Each returned thunk begins generating independently.
    thunks: list[ModelOutputThunk] = []
    for item in items:
        thunk: ModelOutputThunk[str] = await m.ainstruct(
            "Summarise this customer feedback in one sentence: {{text}}",
            user_variables={"text": item},
        )
        thunks.append(thunk)

    # Resolve every thunk together. This preserves fan-out concurrency.
    await wait_for_all_mots(thunks)

    # Once resolved, each thunk can be converted into its final string value.
    return [str(thunk) for thunk in thunks]


feedback_items: list[str] = [
    "The onboarding was confusing and took far too long. Support was helpful once I got through.",
    "The dashboard is clean and intuitive, but exports fail intermittently.",
    "Great reporting features. Setup took a while, but daily use is smooth.",
    "Billing charged me twice. Still waiting for a refund after two weeks.",
]

# Notebook usage:
batch_summaries: list[str] = await summarise_batch(feedback_items)

for index, summary in enumerate(batch_summaries, start=1):
    print(f"{index}. {summary}")

# Script usage:
# if __name__ == "__main__":
#     batch_summaries = asyncio.run(summarise_batch(feedback_items))
#     for index, summary in enumerate(batch_summaries, start=1):
#         print(f"{index}. {summary}")

# Total time is typically dominated by the slowest single call, not the sum of all calls.


## Step 4: Mixing parallel and sequential steps

Real pipelines usually contain both independent and dependent stages. The goal is not to make everything parallel; it is to parallelize only the parts that can run independently, while keeping explicit sequencing where later steps rely on earlier results.

**Key concepts:**

- Independent tasks can start together. In this pipeline, summarisation and issue extraction both depend only on the raw feedback.
- Dependent tasks must wait. Sentiment classification depends on the finished summary, so it starts later.
- This creates a dependency graph: run unrelated work in parallel, then resolve the branch that feeds downstream computation.
- The result is lower latency without sacrificing clarity or correctness.

The code below mirrors the structure of Tutorial 01, but reorganizes execution. Instead of performing each step one after another, it starts the independent branches immediately, waits for the summary branch, and only then launches classification. This is often the best pattern for production pipelines: explicit dependency ordering, minimal idle time. Again, the notebook version uses top-level [`await`](mellea_tutorials/streaming_and_async.ipynb), while the script version uses a guarded [`asyncio.run()`](mellea_tutorials/streaming_and_async.ipynb).

Because this step prints only completed values, it does not need notebook-specific display plumbing. The main notebook difference is still event-loop handling, not output rendering.

In [ ]:
from mellea.core.base import ModelOutputThunk
from mellea.helpers.async_helpers import wait_for_all_mots


import asyncio
from typing import Literal

import mellea
from mellea import generative


@generative
def classify_sentiment(summary: str) -> Literal["positive", "negative", "mixed"]:
    """Classify the overall sentiment of the customer feedback summary."""
    ...


async def analyze_feedback(feedback: str) -> None:
    """Analyze feedback while mixing parallel and sequential dependencies."""
    m: mellea.MelleaSession = mellea.start_session()

    # Start independent work immediately.
    summary_thunk: ModelOutputThunk[str] = await m.ainstruct(
        "Summarise this customer feedback in one sentence: {{text}}",
        user_variables={"text": feedback},
    )
    issues_thunk: ModelOutputThunk[str] = await m.ainstruct(
        "Extract JSON with main_complaint, positive_aspect, and urgency from: {{text}}",
        user_variables={"text": feedback},
    )

    await wait_for_all_mots([summary_thunk, issues_thunk])

    # Sentiment depends on the summary, so resolve that branch first.
    summary: str = str(summary_thunk)

    # Now launch the dependent classification step.
    sentiment_thunk: Literal['positive', 'negative', 'mixed'] = classify_sentiment(m, summary=summary)
    sentiment: str = str(sentiment_thunk)

    # The issues branch has been running in parallel this whole time.
    issues_json: str = str(issues_thunk)

    print(f"Summary:   {summary}")
    print(f"Sentiment: {sentiment}")
    print(f"Issues:    {issues_json}")


# Notebook usage:
await analyze_feedback(
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)

# Script usage:
# if __name__ == "__main__":
#     asyncio.run(
#         analyze_feedback(
#             "The onboarding was confusing and took far too long. "
#             "Support was helpful once I got through."
#         )
#     )

# Output will vary by model and temperature.


## Step 5: Context and concurrency

Context changes the rules. When you attach a [`ChatContext`](mellea_tutorials/streaming_and_async.ipynb), calls share conversational state, so parallel execution can become unsafe or at least ambiguous. That is why the tutorial highlights a warning around using async methods with chat-style multi-turn context.

**Key concepts:**

- [`ChatContext`](mellea_tutorials/streaming_and_async.ipynb) is appropriate when later turns should depend on earlier turns in the same conversation.
- Shared conversational state means ordering matters; parallel requests could race to update or consume that history.
- If you need multi-turn chat behaviour, await each call fully before starting the next.
- For truly parallel generation, prefer independent requests without shared chat context.

This example demonstrates the safe pattern: one call completes, then the next begins. It is still async code, but intentionally sequential because the underlying state is sequential. In notebooks this should also be called with top-level [`await`](mellea_tutorials/streaming_and_async.ipynb), not [`asyncio.run()`](mellea_tutorials/streaming_and_async.ipynb).

As with Steps 3 and 4, there is no token-level UI streaming here. Notebook-specific handling is only needed for coroutine execution, not for output display.

In [ ]:
from mellea.core.base import ModelOutputThunk


from mellea.core.base import ModelOutputThunk


import asyncio

import mellea
from mellea.stdlib.context import ChatContext


async def sequential_chat() -> None:
    """Use ChatContext safely by awaiting each conversational turn in order."""
    # ChatContext stores conversational state, so calls should be resolved sequentially.
    m: mellea.MelleaSession = mellea.start_session(ctx=ChatContext())

    first_turn: ModelOutputThunk[str] = await m.ainstruct("Greet the user and ask what kind of feedback they want to analyse.")
    print(str(first_turn))

    # Only start the next turn after the previous response has been fully resolved.
    second_turn: ModelOutputThunk[str] = await m.ainstruct(
        "The user wants help analysing customer complaints about onboarding. Reply helpfully."
    )
    print(str(second_turn))


# Notebook usage:
await sequential_chat()

# Script usage:
# if __name__ == "__main__":
#     asyncio.run(sequential_chat())

# If you need parallel generation, avoid sharing ChatContext across those requests.


## What you built

| Pattern | What it gives you |
| --- | --- |
| `ainstruct()` / `achat()` / `aact()` | Non-blocking LLM calls |
| `ModelOption.STREAM` + `astream()` | Token-by-token output for responsive UIs |
| `wait_for_all_mots()` | Fan-out: all thunks resolve concurrently |
| Explicit dependency ordering | Sequential where needed, parallel everywhere else |
| `ChatContext` used sequentially | Safe stateful multi-turn conversations |
